# Data Cleaning & Feature Engineering (EDA-Complete, Modeling-Safe)

This notebook prepares the flight delay dataset for both **exploratory data analysis (EDA)** and
**machine learning modeling**, with a clear focus on **data quality**, **interpretability**, and
**target leakage prevention**.

## Purpose
- Build a rich, analysis-ready dataset for EDA and visualization
- Leakage-safe datasets for model training and evaluation

## Outputs

### EDA Dataset
Used exclusively for analysis and visualization:
- `Data_preprocessing_flight_delay.csv`

Includes:
- Temporal features (day, month, quarter, seasonality, weekend flags)
- Geographic features (origin, destination, state-level indicators)
- Route and carrier descriptors
- Traffic, hub, and popularity indicators
- Interaction features for compound pattern analysis

### Modeling Datasets (Leakage-Safe)
Used for machine learning:
- `outputs/flight_delay_train_features.parquet`
- `outputs/flight_delay_test_features.parquet`

Key safeguards:
- Time-based train/test split
- No post-flight information
- Target-derived features computed on training data only

## Target
- `IS_DELAYED` (arrival delay >= 15 minutes)
- `IS_DELAYED_15` is treated as an alias where present

## Feature Groups
- Temporal (day of week, hour, season)
- Geographic (airports, states, hub indicators)
- Route & carrier
- Distance & flight characteristics
- Interaction features (e.g., weekend × summer, hub-to-hub × summer)

## Why This Matters
- Prevents data leakage
- Ensures realistic model evaluation
- Aligns EDA insights with production modeling constraints

In [1]:
import os
import numpy as np
import pandas as pd

from dotenv import load_dotenv

load_dotenv(override=True)

pd.set_option("display.max_columns", None)

### 1) Configuration


In [2]:
DATA_PATH = os.getenv("DATA_PATH")
DATA_PREPROCESSED_DIR = os.getenv("DATA_PREPROCESSED_DIR")
DELAY_THRESHOLD_MIN = 15
TRAIN_FRACTION = 0.80

OUT_DIR = os.getenv("OUT_DIR")
os.makedirs(OUT_DIR, exist_ok=True)

### 2) Load and basic cleaning


In [3]:
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip() for c in df.columns]

df = df.dropna(subset=["ARR_DELAY"]).copy()

# Parse FL_DATE
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"], errors="coerce")
df = df.dropna(subset=["FL_DATE"]).copy()

# Target: arrival delay >= 15 minutes
df["IS_DELAYED"] = (df["ARR_DELAY"] >= DELAY_THRESHOLD_MIN).astype(int)
df["IS_DELAYED_15"] = df["IS_DELAYED"]  # alias

df.shape, df[["FL_DATE", "ARR_DELAY", "IS_DELAYED"]].head()


((147985, 21),
      FL_DATE  ARR_DELAY  IS_DELAYED
 0 2017-07-11      -11.0           0
 1 2018-03-14      -23.0           0
 2 2017-11-12       -9.0           0
 3 2017-08-22      -21.0           0
 4 2018-03-02      -19.0           0)

### 3) Time parsing for DEP_TIME and ARR_TIME (hhmm -> minutes)


In [4]:
def hhmm_to_minutes(x):
    if pd.isna(x):
        return np.nan
    try:
        x = int(x)
        h = x // 100
        m = x % 100
        if h < 0 or h > 23 or m < 0 or m > 59:
            return np.nan
        return h * 60 + m
    except Exception:
        return np.nan

df["DEP_MINUTES"] = df["DEP_TIME"].apply(hhmm_to_minutes)
df["ARR_MINUTES"] = df["ARR_TIME"].apply(hhmm_to_minutes)

df["DEP_HOUR"] = np.floor(df["DEP_MINUTES"] / 60.0)
df["DEP_HOUR_BIN"] = pd.cut(
    df["DEP_HOUR"],
    bins=[-1, 5, 8, 11, 14, 17, 20, 23, 30],
    labels=["0-5", "6-8", "9-11", "12-14", "15-17", "18-20", "21-23", "24+"],
    include_lowest=True
)

df[["DEP_TIME", "DEP_MINUTES", "DEP_HOUR", "DEP_HOUR_BIN", "ARR_TIME", "ARR_MINUTES"]].head()


,DEP_TIME,DEP_MINUTES,DEP_HOUR,DEP_HOUR_BIN,ARR_TIME,ARR_MINUTES
0,538,338.0,5.0,0-5,757,477.0
1,1829,1109.0,18.0,18-20,2108,1268.0
2,1345,825.0,13.0,12-14,1451,891.0
3,1158,718.0,11.0,9-11,1408,848.0
4,522,322.0,5.0,0-5,737,457.0


### 4) Calendar flags and season features


In [5]:
# Day-of-week flags (assumes 1=Mon..7=Sun)
df["IS_MONDAY"] = (df["DAY_OF_WEEK"] == 1).astype(int)
df["IS_FRIDAY"] = (df["DAY_OF_WEEK"] == 5).astype(int)
df["IS_SUNDAY"] = (df["DAY_OF_WEEK"] == 7).astype(int)
df["IS_WEEKEND"] = df["DAY_OF_WEEK"].isin([6, 7]).astype(int)
df["IS_BUSINESS_DAY"] = df["DAY_OF_WEEK"].isin([1, 2, 3, 4, 5]).astype(int)

# Quarter flags (use QUARTER if present, else derive from MONTH)
if "QUARTER" not in df.columns:
    df["QUARTER"] = ((df["MONTH"] - 1) // 3 + 1).astype(int)

df["IS_Q1"] = (df["QUARTER"] == 1).astype(int)
df["IS_Q2"] = (df["QUARTER"] == 2).astype(int)
df["IS_Q3"] = (df["QUARTER"] == 3).astype(int)
df["IS_Q4"] = (df["QUARTER"] == 4).astype(int)

# Season labels (simple month-based definition)
def month_to_season(m):
    if m in [12, 1, 2]:
        return "Winter"
    if m in [3, 4, 5]:
        return "Spring"
    if m in [6, 7, 8]:
        return "Summer"
    return "Fall"

df["SEASON"] = df["MONTH"].astype(int).apply(month_to_season)
df["IS_SUMMER"] = df["SEASON"].eq("Summer").astype(int)
df["IS_WINTER"] = df["SEASON"].eq("Winter").astype(int)

# Holiday season proxy: late Nov + Dec (approximation without holiday calendar)
df["IS_HOLIDAY_SEASON"] = df["MONTH"].isin([11, 12]).astype(int)

df[["MONTH", "QUARTER", "SEASON", "IS_SUMMER", "IS_WINTER", "IS_HOLIDAY_SEASON"]].head()


,MONTH,QUARTER,SEASON,IS_SUMMER,IS_WINTER,IS_HOLIDAY_SEASON
0,7,3,Summer,1,0,0
1,3,1,Spring,0,0,0
2,11,4,Fall,0,0,1
3,8,3,Summer,1,0,0
4,3,1,Spring,0,0,0


### 5) Route and carrier-route features


In [6]:
df["ROUTE"] = df["ORIGIN"].astype(str) + "-" + df["DEST"].astype(str)
df["CARRIER_ROUTE"] = df["UNIQUE_CARRIER"].astype(str) + "_" + df["ROUTE"].astype(str)

df[["UNIQUE_CARRIER", "ORIGIN", "DEST", "ROUTE", "CARRIER_ROUTE"]].head()


,UNIQUE_CARRIER,ORIGIN,DEST,ROUTE,CARRIER_ROUTE
0,DL,LIT,ATL,LIT-ATL,DL_LIT-ATL
1,DL,BOS,ATL,BOS-ATL,DL_BOS-ATL
2,WN,ATL,DAL,ATL-DAL,WN_ATL-DAL
3,EV,ATL,HPN,ATL-HPN,EV_ATL-HPN
4,DL,MSY,ATL,MSY-ATL,DL_MSY-ATL


### 6) Traffic, popularity, and hub features (EDA-focused)

These are computed on the full dataset for descriptive EDA.
For modeling, we will compute train-only versions later.

In [7]:
# Route popularity and airport traffic (full-dataset counts for EDA)
df["ROUTE_POPULARITY"] = df["ROUTE"].map(df["ROUTE"].value_counts())
df["ORIGIN_TRAFFIC"] = df["ORIGIN"].map(df["ORIGIN"].value_counts())
df["DEST_TRAFFIC"] = df["DEST"].map(df["DEST"].value_counts())
df["CARRIER_VOLUME"] = df["UNIQUE_CARRIER"].map(df["UNIQUE_CARRIER"].value_counts())

# Popular route flag: top 10% by route count
route_pop_threshold = df["ROUTE_POPULARITY"].quantile(0.90)
df["IS_POPULAR_ROUTE"] = (df["ROUTE_POPULARITY"] >= route_pop_threshold).astype(int)

# Define hubs: top 10 airports by total traffic (origin+dest)
airport_total = pd.concat([df["ORIGIN"], df["DEST"]]).value_counts()
hub_airports = set(airport_total.head(10).index.astype(str))

df["IS_HUB_ORIGIN"] = df["ORIGIN"].astype(str).isin(hub_airports).astype(int)
df["IS_HUB_DEST"] = df["DEST"].astype(str).isin(hub_airports).astype(int)
df["IS_HUB_TO_HUB"] = ((df["IS_HUB_ORIGIN"] == 1) & (df["IS_HUB_DEST"] == 1)).astype(int)

# Busy airports: above median traffic within origin/dest
origin_busy_thr = df["ORIGIN_TRAFFIC"].median()
dest_busy_thr = df["DEST_TRAFFIC"].median()
df["IS_BUSY_ORIGIN"] = (df["ORIGIN_TRAFFIC"] >= origin_busy_thr).astype(int)
df["IS_BUSY_DEST"] = (df["DEST_TRAFFIC"] >= dest_busy_thr).astype(int)

df[["ROUTE_POPULARITY","IS_POPULAR_ROUTE","ORIGIN_TRAFFIC","DEST_TRAFFIC","IS_HUB_ORIGIN","IS_HUB_DEST","IS_HUB_TO_HUB","IS_BUSY_ORIGIN","IS_BUSY_DEST","CARRIER_VOLUME"]].head()


,ROUTE_POPULARITY,IS_POPULAR_ROUTE,ORIGIN_TRAFFIC,DEST_TRAFFIC,IS_HUB_ORIGIN,IS_HUB_DEST,IS_HUB_TO_HUB,IS_BUSY_ORIGIN,IS_BUSY_DEST,CARRIER_VOLUME
0,417,0,417,72704,0,1,0,0,1,95931
1,1386,1,1484,72704,1,1,1,0,1,95931
2,696,0,72525,696,1,0,0,1,0,16370
3,419,0,72525,419,1,0,0,1,0,9922
4,1123,0,1123,72704,0,1,0,0,1,95931


### 7) Distance categories and interactions


In [8]:
# Distance categories (miles)
df["DISTANCE_CAT"] = pd.cut(
    df["DISTANCE"],
    bins=[-1, 750, 1500, 1000000],
    labels=["SHORT", "MEDIUM", "LONG"]
)

df["IS_SHORT_HAUL"] = (df["DISTANCE_CAT"] == "SHORT").astype(int)
df["IS_MEDIUM_HAUL"] = (df["DISTANCE_CAT"] == "MEDIUM").astype(int)
df["IS_LONG_HAUL"] = (df["DISTANCE_CAT"] == "LONG").astype(int)

# Normalized distance (z-score)
dist_mean = df["DISTANCE"].mean()
dist_std = df["DISTANCE"].std(ddof=0) if df["DISTANCE"].std(ddof=0) != 0 else 1.0
df["DISTANCE_NORMALIZED"] = (df["DISTANCE"] - dist_mean) / dist_std

# Interaction flags (as per your column list)
df["WEEKEND_SUMMER"] = ((df["IS_WEEKEND"] == 1) & (df["IS_SUMMER"] == 1)).astype(int)
df["FRIDAY_SUMMER"] = ((df["IS_FRIDAY"] == 1) & (df["IS_SUMMER"] == 1)).astype(int)
df["MONDAY_WINTER"] = ((df["IS_MONDAY"] == 1) & (df["IS_WINTER"] == 1)).astype(int)
df["HUB_TO_HUB_SUMMER"] = ((df["IS_HUB_TO_HUB"] == 1) & (df["IS_SUMMER"] == 1)).astype(int)
df["POPULAR_ROUTE_WEEKEND"] = ((df["IS_POPULAR_ROUTE"] == 1) & (df["IS_WEEKEND"] == 1)).astype(int)
df["LONG_HAUL_WINTER"] = ((df["IS_LONG_HAUL"] == 1) & (df["IS_WINTER"] == 1)).astype(int)

df[["DISTANCE","DISTANCE_CAT","IS_SHORT_HAUL","IS_MEDIUM_HAUL","IS_LONG_HAUL","DISTANCE_NORMALIZED","WEEKEND_SUMMER","FRIDAY_SUMMER","MONDAY_WINTER","HUB_TO_HUB_SUMMER","POPULAR_ROUTE_WEEKEND","LONG_HAUL_WINTER"]].head()


,DISTANCE,DISTANCE_CAT,IS_SHORT_HAUL,IS_MEDIUM_HAUL,IS_LONG_HAUL,DISTANCE_NORMALIZED,WEEKEND_SUMMER,FRIDAY_SUMMER,MONDAY_WINTER,HUB_TO_HUB_SUMMER,POPULAR_ROUTE_WEEKEND,LONG_HAUL_WINTER
0,453,SHORT,1,0,0,-0.436952,0,0,0,0,0,0
1,946,MEDIUM,0,1,0,0.613493,0,0,0,0,0,0
2,721,SHORT,1,0,0,0.134081,0,0,0,0,0,0
3,780,MEDIUM,0,1,0,0.259794,0,0,0,0,0,0
4,425,SHORT,1,0,0,-0.496612,0,0,0,0,0,0


### 8) Save full EDA dataset (all columns)

This file is intended for EDA visualizations and includes full engineered column set.

In [9]:
df.to_csv(DATA_PREPROCESSED_DIR, index=False)

### 9) Create leakage-safe modeling datasets

Computing target encodings (risk) and train-only count features **after** time splitting.

In [10]:
# Time-based split
df_sorted = df.sort_values("FL_DATE").reset_index(drop=True)
split_idx = int(len(df_sorted) * TRAIN_FRACTION)

train_df = df_sorted.iloc[:split_idx].copy()
test_df  = df_sorted.iloc[split_idx:].copy()

print("Train date range:", train_df["FL_DATE"].min(), "->", train_df["FL_DATE"].max())
print("Test  date range:", test_df["FL_DATE"].min(), "->", test_df["FL_DATE"].max())
print("Delay rate train:", round(train_df["IS_DELAYED"].mean(), 4))
print("Delay rate test: ", round(test_df["IS_DELAYED"].mean(), 4))


Train date range: 2017-05-01 00:00:00 -> 2018-02-22 00:00:00
Test  date range: 2018-02-23 00:00:00 -> 2018-04-30 00:00:00
Delay rate train: 0.1549
Delay rate test:  0.1291


In [11]:
def add_smoothed_target_encoding(train_df, test_df, col, target="IS_DELAYED", smoothing=50):
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(["mean", "count"])
    enc = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
    train_df[f"{col}_RISK"] = train_df[col].map(enc).fillna(global_mean)
    test_df[f"{col}_RISK"]  = test_df[col].map(enc).fillna(global_mean)
    return train_df, test_df

for col in ["ORIGIN", "DEST", "UNIQUE_CARRIER", "ROUTE", "CARRIER_ROUTE"]:
    if col in train_df.columns:
        train_df, test_df = add_smoothed_target_encoding(train_df, test_df, col, target="IS_DELAYED", smoothing=50)

risk_cols = [c for c in train_df.columns if c.endswith("_RISK")]
risk_cols


['ORIGIN_RISK',
 'DEST_RISK',
 'UNIQUE_CARRIER_RISK',
 'ROUTE_RISK',
 'CARRIER_ROUTE_RISK']

In [12]:
def add_count_feature(train_df, test_df, col, prefix="TRAIN"):
    counts = train_df[col].value_counts()
    train_df[f"{prefix}_{col}_COUNT"] = train_df[col].map(counts).fillna(0).astype(int)
    test_df[f"{prefix}_{col}_COUNT"]  = test_df[col].map(counts).fillna(0).astype(int)
    return train_df, test_df

for col in ["ORIGIN", "DEST", "ROUTE", "UNIQUE_CARRIER", "CARRIER_ROUTE"]:
    if col in train_df.columns:
        train_df, test_df = add_count_feature(train_df, test_df, col, prefix="TRAIN")

count_cols = [c for c in train_df.columns if c.endswith("_COUNT")]
count_cols


['TRAIN_ORIGIN_COUNT',
 'TRAIN_DEST_COUNT',
 'TRAIN_ROUTE_COUNT',
 'TRAIN_UNIQUE_CARRIER_COUNT',
 'TRAIN_CARRIER_ROUTE_COUNT']

### 10) Assemble modeling feature tables and save

Excluding raw `FL_DATE` from features later in modeling, but keep it for reporting and keeping all EDA features plus leakage-safe risk and train-only count columns.

In [13]:
TARGET = "IS_DELAYED_15"  # alias, same as IS_DELAYED

# Build modeling table: everything except ARR_DELAY is allowed as input (ARR_DELAY is direct label leakage)
exclude = ["ARR_DELAY"]
model_feature_cols = [c for c in train_df.columns if c not in exclude]

train_out = train_df[model_feature_cols].copy()
test_out  = test_df[model_feature_cols].copy()

train_path = os.path.join(OUT_DIR, "flight_delay_train_features.parquet")
test_path  = os.path.join(OUT_DIR, "flight_delay_test_features.parquet")
final_path = os.path.join(OUT_DIR, "flight_delay_features_final.parquet")

train_out.to_parquet(train_path, index=False)
test_out.to_parquet(test_path, index=False)
pd.concat([train_out, test_out], axis=0).sort_values("FL_DATE").reset_index(drop=True).to_parquet(final_path, index=False)

### 11) Schema check


In [14]:
expected_cols = [
 'YEAR','QUARTER','MONTH','DAY_OF_MONTH','DAY_OF_WEEK','FL_DATE',
 'UNIQUE_CARRIER','ORIGIN','ORIGIN_CITY_NAME','ORIGIN_STATE_ABR',
 'DEST','DEST_CITY_NAME','DEST_STATE_ABR','DEP_TIME','ARR_TIME',
 'ARR_DELAY','AIR_TIME','DISTANCE','DISTANCE_GROUP','IS_DELAYED',
 'IS_WEEKEND','IS_MONDAY','IS_FRIDAY','IS_SUNDAY','IS_BUSINESS_DAY',
 'SEASON','IS_SUMMER','IS_WINTER','IS_HOLIDAY_SEASON','IS_Q1','IS_Q2','IS_Q3','IS_Q4',
 'ROUTE','CARRIER_ROUTE','ROUTE_POPULARITY','IS_POPULAR_ROUTE','ORIGIN_TRAFFIC','DEST_TRAFFIC',
 'IS_HUB_ORIGIN','IS_HUB_DEST','IS_HUB_TO_HUB','IS_BUSY_ORIGIN','IS_BUSY_DEST',
 'CARRIER_VOLUME','DISTANCE_CAT','IS_SHORT_HAUL','IS_MEDIUM_HAUL','IS_LONG_HAUL',
 'DISTANCE_NORMALIZED','WEEKEND_SUMMER','FRIDAY_SUMMER','MONDAY_WINTER','HUB_TO_HUB_SUMMER',
 'POPULAR_ROUTE_WEEKEND','LONG_HAUL_WINTER'
]

missing = [c for c in expected_cols if c not in df.columns]
extra = [c for c in df.columns if c not in expected_cols]

print("Missing expected EDA columns:", missing)
print("Extra columns in EDA dataset (OK):", extra[:20], "..." if len(extra) > 20 else "")
print("EDA dataset columns:", len(df.columns))


Missing expected EDA columns: []
Extra columns in EDA dataset (OK): ['IS_DELAYED_15', 'DEP_MINUTES', 'ARR_MINUTES', 'DEP_HOUR', 'DEP_HOUR_BIN'] 
EDA dataset columns: 61
